# 02 计算盗垒成功率

## 这一步做了什么
读取 Day 1 抓取的逐球数据，按缺失值类型逐列清洗，识别所有与盗垒相关的事件，
计算当日全联盟的盗垒成功率，并分球队、分跑者汇总。

## 数据从哪来
- 输入：`data/raw/2025-06-14_statcast.csv`（Day 1 产出，逐球粒度）
- 盗垒事件来自 `events` 列，该列**只在每个打席的最后一球有值**

## 结果是什么
- 全联盟当日盗垒成功率（含 95% Wilson 置信区间）
- 分球队表：`data/processed/2025-06-14_sb_by_team.csv`
- 盗垒明细表：`data/processed/2025-06-14_sb_events.csv`

## 统计口径
成功率 = 盗垒成功次数 / 盗垒尝试次数。分母 = 盗垒成功 + 盗垒失败(CS) + 牵制致CS，
不含纯牵制出局(PO)与"其他进垒"。


In [1]:
# ===== 读取 Day 1 的数据 =====
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(marker: str = "requirements.txt") -> Path:
    """从当前工作目录逐级向上找，直到找到含指定文件的目录。
    这样无论 notebook 从哪个目录启动，路径都不会错。
    """
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    return p

ROOT = find_project_root()
DATE = "2025-06-14"
RAW_PATH = ROOT / "data" / "raw" / f"{DATE}_statcast.csv"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH, low_memory=False)
df["game_date"] = pd.to_datetime(df["game_date"])

print("项目根目录:", ROOT)
print("读取形状:", df.shape)
print("比赛场数:", df["game_pk"].nunique())
df.head(3)


项目根目录: C:\Users\GOOD\PycharmProjects\Baseball-Analytics
读取形状: (4435, 26)
比赛场数: 15


,game_date,pitcher,batter,release_speed,events,launch_speed,launch_angle,game_pk,game_type,home_team,...,on_3b,at_bat_number,pitch_number,description,type,bb_type,stand,p_throws,pitcher_name,batter_name
0,2025-06-14,669276,687597,87.9,strikeout,NaN,NaN,777523,R,ATL,...,NaN,66,5,swinging_strike_blocked,S,NaN,R,L,Dylan Lee,Jordan Beck
1,2025-06-14,669276,687597,87.9,NaN,NaN,NaN,777523,R,ATL,...,NaN,66,4,ball,B,NaN,R,L,Dylan Lee,Jordan Beck
2,2025-06-14,669276,687597,86.3,NaN,NaN,NaN,777523,R,ATL,...,NaN,66,3,foul,S,NaN,R,L,Dylan Lee,Jordan Beck


## 缺失值清洗

逐球数据的缺失有明确含义，**不能统一删行或统一填 0**。

| 列 | 缺失原因 | 处理方式 |
|---|---|---|
| `events` | 该球不是打席最后一球 | 保留缺失；分析时按需筛选非空 |
| `launch_speed` / `launch_angle` | 球没被打进场（三振、保送、界外） | **保留 NaN，禁止填 0** |
| `on_1b` / `on_2b` / `on_3b` | 该垒无人 | 保留缺失，另建是否有人上垒的 0/1 列 |
| `outs_when_up` | 极少缺失 | 填 0，并记录填补条数 |
| `bb_type` | 球没被打进场 | 保留缺失 |
| `pitcher` / `batter` | 无 | 转为可空整型 Int64 |

关键原则：**对 `launch_speed` 填 0 会把"没击球"混进"击球初速为 0"，直接污染后续所有平均值**。


In [2]:
# ===== 缺失值清洗 =====
n0 = len(df)

# 1) 删除完全重复的行（抓取重试或拼接时可能产生）
df = df.drop_duplicates().reset_index(drop=True)

# 2) 出局数：应为 0/1/2，缺失填 0 并报告条数
n_out_missing = df["outs_when_up"].isna().sum()
df["outs_when_up"] = df["outs_when_up"].fillna(0).astype(int)

# 3) ID 列转可空整型，避免 NaN 导致 astype(int) 报错
for col in ["pitcher", "batter"]:
    if col in df.columns:
        df[col] = df[col].astype("Int64")

# 4) launch_speed / launch_angle：缺失代表未击球，保持 NaN 不动
#    这里只是明确声明"不处理"，防止以后误填

# 5) 由垒上跑者列派生 0/1 标记，供 Day 3 构造 24 状态使用
for base in ["on_1b", "on_2b", "on_3b"]:
    df[f"has_{base}"] = df[base].notna().astype(int)

print(f"原始行数 {n0} → 去重后 {len(df)}（删除 {n0 - len(df)} 行）")
print(f"outs_when_up 缺失填补 {n_out_missing} 行")

# 6) 数据完整性检查：同一场比赛同一打席同一球序不应重复
dup_key = ["game_pk", "at_bat_number", "pitch_number"]
dup_n = df.duplicated(subset=dup_key).sum()
print(f"球序重复记录: {dup_n} 行")

# 7) 出局数取值范围检查
print("outs_when_up 取值:", sorted(df["outs_when_up"].unique()))


原始行数 4435 → 去重后 4435（删除 0 行）
outs_when_up 缺失填补 0 行
球序重复记录: 0 行
outs_when_up 取值: [np.int64(0), np.int64(1), np.int64(2)]


In [3]:
# ===== 先列出当天所有与盗垒/牵制相关的事件名 =====
# 不靠记忆背事件名，直接让数据说话——事件命名在不同年份可能变化
events_unique = df["events"].dropna().unique()

steal_like = sorted([e for e in events_unique if ("steal" in e) or ("pickoff" in e)])
print("当天与盗垒/牵制相关的事件：")
for e in steal_like:
    print("  -", e, " 出现", (df["events"] == e).sum(), "次")

if not steal_like:
    print("当天没有任何盗垒或牵制事件 —— 先换一个日期再试")


当天与盗垒/牵制相关的事件：
当天没有任何盗垒或牵制事件 —— 先换一个日期再试


## 事件分类

根据上一步打印的实际情况，把事件分成三组：

- **SB（盗垒成功）**：`stolen_base_2b`、`stolen_base_3b`、`stolen_base_home`
- **CS（盗垒失败）**：`caught_stealing_*` 与 `pickoff_caught_stealing_*`
- **PO（纯牵制出局）**：`pickoff_1b`、`pickoff_2b`、`pickoff_3b`，**不计入分母**

如果上一步发现当天出现了表里没列的事件名，需要手工判断它归哪一组。


In [4]:
# ===== 事件分类 + 生成盗垒明细表 =====
SB_EVENTS = {
    "stolen_base_2b": 2,
    "stolen_base_3b": 3,
    "stolen_base_home": 4,
}
CS_EVENTS = {
    "caught_stealing_2b": 2,
    "caught_stealing_3b": 3,
    "caught_stealing_home": 4,
    "pickoff_caught_stealing_2b": 2,
    "pickoff_caught_stealing_3b": 3,
    "pickoff_caught_stealing_home": 4,
}
PO_EVENTS = ["pickoff_1b", "pickoff_2b", "pickoff_3b"]

# 只保留打席终结球（events 非空的行），盗垒事件只会出现在这里
ev = df[df["events"].notna()].copy()

sb = ev[ev["events"].isin(SB_EVENTS)].copy()
cs = ev[ev["events"].isin(CS_EVENTS)].copy()
po = ev[ev["events"].isin(PO_EVENTS)].copy()

# 列出分类后仍未覆盖的疑似相关事件，人工核对
known = set(SB_EVENTS) | set(CS_EVENTS) | set(PO_EVENTS)
unclassified = sorted([e for e in steal_like if e not in known])
if unclassified:
    print("⚠ 以下事件未被分类，请人工判断：", unclassified)

print(f"盗垒成功 SB = {len(sb)}")
print(f"盗垒失败 CS = {len(cs)}")
print(f"纯牵制出局 PO = {len(po)}（不计入成功率分母）")
print(f"盗垒尝试 = {len(sb) + len(cs)}")


盗垒成功 SB = 0
盗垒失败 CS = 0
纯牵制出局 PO = 0（不计入成功率分母）
盗垒尝试 = 0


In [5]:
# ===== 计算总体盗垒成功率 =====
def wilson_ci(k: int, n: int, z: float = 1.96) -> tuple:
    """Wilson 得分区间。相比正态近似，在小样本和 p 接近 0/1 时更可靠。
    k 成功次数，n 总次数，z=1.96 对应 95% 置信水平。
    """
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))

n_sb, n_cs = len(sb), len(cs)
n_att = n_sb + n_cs
sb_rate = n_sb / n_att if n_att > 0 else np.nan
ci_low, ci_high = wilson_ci(n_sb, n_att)

print(f"日期: {DATE}")
print(f"盗垒成功率 = {n_sb}/{n_att} = {sb_rate:.1%}")
print(f"95% Wilson 区间: [{ci_low:.1%}, {ci_high:.1%}]")
print(f"（作为参照：近年大联盟整体盗垒成功率约 78%–81%）")


日期: 2025-06-14
盗垒成功率 = 0/0 = nan%
95% Wilson 区间: [nan%, nan%]
（作为参照：近年大联盟整体盗垒成功率约 78%–81%）


In [6]:
# ===== 分球队统计 =====
# 进攻方球队：上半局是客队，下半局是主队
def add_batting_team(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d["batting_team"] = np.where(d["inning_topbot"].eq("Top"),
                                 d["away_team"], d["home_team"])
    return d

sb_t = add_batting_team(sb)
cs_t = add_batting_team(cs)

team_stats = pd.DataFrame({
    "SB": sb_t.groupby("batting_team").size(),
    "CS": cs_t.groupby("batting_team").size(),
}).fillna(0).astype(int)

team_stats["attempts"] = team_stats["SB"] + team_stats["CS"]
team_stats["sb_rate"] = (team_stats["SB"] / team_stats["attempts"]).round(3)
team_stats = team_stats.sort_values(["attempts", "SB"], ascending=False)

print(team_stats.to_string())
team_stats.to_csv(OUT_DIR / f"{DATE}_sb_by_team.csv", encoding="utf-8-sig")
print(f"\n已保存: {OUT_DIR / f'{DATE}_sb_by_team.csv'}")


Empty DataFrame
Columns: [SB, CS, attempts, sb_rate]
Index: []

已保存: C:\Users\GOOD\PycharmProjects\Baseball-Analytics\data\processed\2025-06-14_sb_by_team.csv


In [7]:
# ===== 生成盗垒明细表 =====
# Statcast 没有直接给出"谁盗的垒"，但可以从跑者所在垒位反推：
#   盗二垒的跑者原本在一垒 → on_1b
#   盗三垒的跑者原本在二垒 → on_2b
#   盗本垒的跑者原本在三垒 → on_3b
BASE_COL = {2: "on_1b", 3: "on_2b", 4: "on_3b"}

def build_detail(d: pd.DataFrame, event_map: dict, success: bool) -> pd.DataFrame:
    out = d.copy()
    out["target_base"] = out["events"].map(event_map)
    out["runner_id"] = [row[BASE_COL[b]] for row, b in zip(out.to_dict("records"), out["target_base"])]
    out["success"] = int(success)
    cols = ["game_date", "game_pk", "batting_team", "runner_id", "events",
            "target_base", "success", "outs_when_up"]
    return out[cols]

sb_detail = build_detail(sb_t, SB_EVENTS, True)
cs_detail = build_detail(cs_t, CS_EVENTS, False)
detail = pd.concat([sb_detail, cs_detail]).sort_values(["game_pk", "game_date"])

print(f"盗垒明细共 {len(detail)} 条")
print(detail.head(10).to_string())

detail.to_csv(OUT_DIR / f"{DATE}_sb_events.csv", index=False, encoding="utf-8-sig")
print(f"已保存: {OUT_DIR / f'{DATE}_sb_events.csv'}")

# 分跑者汇总（只列有 2 次及以上尝试的，单次样本没有解释意义）
runner_stats = (detail.groupby("runner_id")
                .agg(attempts=("success", "size"), SB=("success", "sum")))
runner_stats["CS"] = runner_stats["attempts"] - runner_stats["SB"]
runner_stats["sb_rate"] = (runner_stats["SB"] / runner_stats["attempts"]).round(3)
print("\n当日有多于 1 次盗垒尝试的跑者：")
print(runner_stats[runner_stats["attempts"] >= 2].sort_values("attempts", ascending=False).to_string())


盗垒明细共 0 条
Empty DataFrame
Columns: [game_date, game_pk, batting_team, runner_id, events, target_base, success, outs_when_up]
Index: []
已保存: C:\Users\GOOD\PycharmProjects\Baseball-Analytics\data\processed\2025-06-14_sb_events.csv

当日有多于 1 次盗垒尝试的跑者：
Empty DataFrame
Columns: [attempts, SB, CS, sb_rate]
Index: []


In [8]:
# ===== 保存当日汇总 =====
summary = pd.DataFrame([{
    "date": DATE,
    "games": int(df["game_pk"].nunique()),
    "pitches": len(df),
    "SB": n_sb,
    "CS": n_cs,
    "attempts": n_att,
    "sb_rate": round(sb_rate, 4),
    "ci_low": round(ci_low, 4),
    "ci_high": round(ci_high, 4),
}])
summary.to_csv(OUT_DIR / f"{DATE}_sb_summary.csv", index=False, encoding="utf-8-sig")

print("=== 当日结论 ===")
print(f"{DATE} 共 {summary.loc[0, 'games']} 场比赛，{summary.loc[0, 'pitches']} 次投球。")
print(f"发生盗垒尝试 {n_att} 次：成功 {n_sb} 次、失败 {n_cs} 次。")
print(f"盗垒成功率 {sb_rate:.1%}，95% 置信区间 [{ci_low:.1%}, {ci_high:.1%}]。")
print(f"\n注意：单日样本仅 {n_att} 次尝试，该数值不能代表球队真实水平。")


=== 当日结论 ===
2025-06-14 共 15 场比赛，4435 次投球。
发生盗垒尝试 0 次：成功 0 次、失败 0 次。
盗垒成功率 nan%，95% 置信区间 [nan%, nan%]。

注意：单日样本仅 0 次尝试，该数值不能代表球队真实水平。
